In [1]:
# ======================================================================
# DUBLIN RENT (processed): Dublin rents only + location columns
# ======================================================================
# Input:  dataSet\processed_dataset\rent_index_dataset\rent_2021_onwards.csv
# Output: just_dublin_dataset\just_dublin_processed_dataset\rent\rent_dublin.csv
#
# Rent locations come in 3 kinds:
#   'Dublin'              -> the whole county
#   'Dublin 12'           -> a whole postal district
#   'Crumlin, Dublin 12'  -> a smaller area inside a district
#   'Swords, Dublin'      -> an area with no district number
#
# Needs 00_dublin_location_lookup.ipynb to be run first.
# ======================================================================

import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project'

rent_file = base + r'\dataSet\processed_dataset\rent_index_dataset\rent_2021_onwards.csv'
dublin_processed = base + r'\just_dublin_dataset\just_dublin_processed_dataset'
save_folder = dublin_processed + r'\rent'

os.makedirs(save_folder, exist_ok=True)

In [2]:
rent = pd.read_csv(rent_file)
print('All of Ireland:', rent.shape)

All of Ireland: (80292, 9)


In [3]:
# Keeping only Dublin rents

rent = rent[rent['county'] == 'Dublin']
print('Dublin only:', rent.shape)
print(rent['location'].nunique(), 'Dublin locations')

Dublin only: (27319, 9)
155 Dublin locations


In [4]:
# District: the part after the comma ('Crumlin, Dublin 12' -> 'Dublin 12'),
# or the whole name when there is no comma ('Dublin 12').

lookup = pd.read_csv(dublin_processed + r'\dublin_location_lookup.csv')
district_list = list(lookup['dublin_district'])

district_part = []
for loc in rent['location']:
    if ',' in loc:
        district_part.append(loc.split(',')[1].strip())
    else:
        district_part.append(loc.strip())

rent['dublin_district'] = district_part
rent.loc[~rent['dublin_district'].isin(district_list), 'dublin_district'] = 'Dublin (no district)'

rent['dublin_district'].value_counts()

dublin_district
Dublin (no district)    5941
Dublin 4                1962
Dublin 15               1943
Dublin 6                1930
Dublin 8                1893
Dublin 3                1808
Dublin 9                1276
Dublin 18               1205
Dublin 7                1146
Dublin 16                956
Dublin 12                933
Dublin 24                883
Dublin 1                 803
Dublin 14                684
Dublin 13                661
Dublin 2                 606
Dublin 6W                603
Dublin 5                 509
Dublin 11                472
Dublin 22                353
Dublin 10                290
Dublin 20                254
Dublin 17                208
Name: count, dtype: int64

In [5]:
# Location level, so it is clear what each row describes:
#   county   -> 'Dublin' (all of Dublin)
#   district -> exactly a postal district ('Dublin 12')
#   local    -> an area inside a district or without a district

rent['location_level'] = 'local'
rent.loc[rent['location'] == 'Dublin', 'location_level'] = 'county'
rent.loc[rent['location'].isin(district_list), 'location_level'] = 'district'

# the whole-county row is not one district
rent.loc[rent['location_level'] == 'county', 'dublin_district'] = np.nan

print(rent['location_level'].value_counts())

location_level
local       24865
district     1629
county        825
Name: count, dtype: int64


In [6]:
# Which districts have their own district-level rent rows?
# (some districts only appear as smaller areas, e.g. 'Clondalkin, Dublin 22')

district_rows = rent[rent['location_level'] == 'district']
print('Districts with their own rows:', district_rows['dublin_district'].nunique(), 'of', len(district_list) - 1)

missing = []
for d in district_list:
    if d != 'Dublin (no district)' and d not in list(district_rows['dublin_district']):
        missing.append(d)
print('Districts without their own rows:', missing)

Districts with their own rows: 21 of 22
Districts without their own rows: ['Dublin 22']


In [7]:
# Adding district_id and the higher levels from the lookup.
# local_authority is left empty (the rent index doesn't use councils for Dublin).

rent = rent.merge(lookup[['dublin_district', 'district_id', 'county_id', 'province', 'nation']],
                  on='dublin_district', how='left')

rent['county_id'] = lookup['county_id'].values[0]   # also for the whole-county row
rent['province'] = 'Leinster'
rent['nation'] = 'Ireland'
rent['local_authority'] = np.nan

rent[['location', 'location_level', 'dublin_district', 'district_id', 'county',
      'province', 'nation']].drop_duplicates().head(10)

,location,location_level,dublin_district,district_id,county,province,nation
0,Dublin,county,NaN,NaN,Dublin,Leinster,Ireland
1,"Balbriggan, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
2,"Blackrock, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
3,"Booterstown, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
4,"Dalkey, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
5,"Donabate, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
6,"Dun Laoghaire, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
7,"Glenageary, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
8,"Howth, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland
9,"Killiney, Dublin",local,Dublin (no district),23.0,Dublin,Leinster,Ireland


In [8]:
rent.to_csv(save_folder + r'\rent_dublin.csv', index=False)
print('Saved', rent.shape, 'to', save_folder + r'\rent_dublin.csv')

Saved (27319, 15) to C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\just_dublin_dataset\just_dublin_processed_dataset\rent\rent_dublin.csv
